In [5]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely import wkb

In [3]:
import seaborn as sns

# Research Area
- City of London
- Westminster
- Hackney
- Camden

# Time Range
- First period: 2016-06-30, 2016-12-31, **2017-06-30**, 2017-12-31, 2018-06-30, 2019-06-30
- Second period: 2022-06-30, 2022-12-31, **2023-06-30**, 2023-12-31, 2024-06-30, 2025-06-30

# First Period Analysis
## Import the data

In [4]:
citylondon_17 = pd.read_csv('data_source/period-2010-to-2017-E09000001-city of london.csv')
camden_17 = pd.read_csv('data_source/period-2010-to-2017-E09000007-camden.csv')
hackney_17 = pd.read_csv('data_source/period-2010-to-2017-E09000012-hackney.csv')
westminster_17 = pd.read_csv('data_source/period-2010-to-2017-E09000033-westminster.csv')

/tmp/ipykernel_39155/722522668.py:1: DtypeWarning: Columns (5,10,11,19,21) have mixed types. Specify dtype option on import or set low_memory=False.
  citylondon_17 = pd.read_csv('data_source/period-2010-to-2017-E09000001-city of london.csv')
/tmp/ipykernel_39155/722522668.py:3: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  hackney_17 = pd.read_csv('data_source/period-2010-to-2017-E09000012-hackney.csv')
/tmp/ipykernel_39155/722522668.py:4: DtypeWarning: Columns (5,10,11,19,21) have mixed types. Specify dtype option on import or set low_memory=False.
  westminster_17 = pd.read_csv('data_source/period-2010-to-2017-E09000033-westminster.csv')


In [6]:
citylondon_17.columns

Index(['filter_period', 'billing_authority_name', 'geocode', 'uarn',
       'billing_reference', 'account_name', 'account_start_date',
       'searchable_address', 'postcode_id', 'geometry', 'occupation_state',
       'occupation_date', 'category_id', 'primary_description',
       'category_subgroup', 'category_group', 'rates_payable',
       'rateable_value', 'total_floor_area', 'unit_of_measure', 'from_date',
       'record_date', 'series', 'epoch'],
      dtype='object')

In [8]:
# Cleaning
citylondon_17N = citylondon_17[citylondon_17['rateable_value'] > 0]  # Remove outliers
citylondon_17N = citylondon_17N.dropna(subset=['rateable_value'])

camden_17N = camden_17[camden_17['rateable_value'] > 0]
camden_17N = camden_17N.dropna(subset=['rateable_value'])

hackney_17N = hackney_17[hackney_17['rateable_value'] > 0]
hackney_17N = hackney_17N.dropna(subset=['rateable_value'])

westminster_17N = westminster_17[westminster_17['rateable_value'] > 0]
westminster_17N = westminster_17N.dropna(subset=['rateable_value'])

# Convert WKB geometry to Shapely objects
citylondon_17N['geom'] = citylondon_17N['geometry'].apply(lambda x: wkb.loads(x, hex=True))
camden_17N['geom'] = camden_17N['geometry'].apply(lambda x: wkb.loads(x, hex=True))
hackney_17N['geom'] = hackney_17N['geometry'].apply(lambda x: wkb.loads(x, hex=True))
westminster_17N['geom'] = westminster_17N['geometry'].apply(lambda x: wkb.loads(x, hex=True))

In [9]:
time_points = {
    '2016-06-30': 'pre_12m',
    '2016-12-31': 'pre_6m', 
    '2017-06-30': 'revaluation',
    '2017-12-31': 'post_6m',
    '2018-06-30': 'post_12m',
    '2019-06-30': 'post_24m'
}
# Extract data from all time points
def extract_all_timepoints(df):
    timepoints = {
        'pre_12m': df[(df['series'] == 2010) & (df['epoch'] == 21)],
        'pre_6m': df[(df['series'] == 2010) & (df['epoch'] == 23)],
        'revaluation': df[(df['series'] == 2017) & (df['epoch'] == 2)],
        'post_6m': df[(df['series'] == 2017) & (df['epoch'] == 4)],
        'post_12m': df[(df['series'] == 2017) & (df['epoch'] == 7)],
        'post_24m': df[(df['series'] == 2017) & (df['epoch'] == 13)]
    }
    return timepoints

## Dynamic Bunching Analysis
### Function
- 检测门槛聚集效应：量化企业在税率门槛附近的分布扭曲程度
- 追踪时间演变：观察bunching效应在重估前后的动态变化
- 识别bracket creep：证明税率门槛确实影响企业行为

### 分析机制
- 对比分布密度：比较门槛前**1000英镑**区间 vs 门槛后1000英镑区间的企业数量 \
ratio = (number of businesses in £1000 range before threshold) / (number of businesses in £1000 range after threshold)
- 计算扭曲比例：**ratio < 1**表示门槛前企业"异常稀少"，存在bunching \
distortion_pct = (ratio - 1) * 100
- 量化扭曲程度：distortion_pct显示偏离正常分布的百分比



### <font color ='#DE3163'>可以尝试不同窗口大小：
- window=500   更精细的分析
- window=2000  更宽泛的范围  
- window=1500  中等范围

In [11]:
def dynamic_bunching_analysis(timepoints_dict, thresholds=[15000], window=1000):
    results = []
    
    for period, df in timepoints_dict.items():
        if len(df) == 0:
            continue
            
        df_clean = df.dropna(subset=['rateable_value'])
        
        for threshold in thresholds:
            before = len(df_clean[(df_clean['rateable_value'] >= threshold-window) & 
                                 (df_clean['rateable_value'] < threshold)])
            after = len(df_clean[(df_clean['rateable_value'] >= threshold) & 
                                (df_clean['rateable_value'] <= threshold+window)])
            
            ratio = before / max(after, 1)
            distortion = (ratio - 1) * 100
            
            results.append({
                'period': period,
                'threshold': threshold,
                'before_count': before,
                'after_count': after,
                'ratio': ratio,
                'distortion_pct': distortion
            })
    
    return pd.DataFrame(results)

## Survival analysis
- "哪些企业因为bracket creep而被迫退出市场？"➡️识别可能受bracket creep影响的企业群体
- 如果门槛附近企业退出率更高，证明bracket creep造成了额外压力

### 核心功能
- 追踪企业去向：监测同一批企业在不同时间点是否仍然存在 ➡️ 退出是立即发生还是延迟发生？
- 计算退出率`survival_rate`：量化有多少企业在重估后"消失"了
- 对比分析：比较门槛附近企业 `survival_rate_near_threshold` vs 整体企业的生存表现 `survival_rate_total`

### 分析机制
- 基准建立：以重估前12个月的企业为起始群体
- 生存追踪：检查这些企业的UARN在后续时间点是否还存在
- 分组对比：
整体生存率：所有企业的存活比例 \
门槛附近生存率：15000门槛±window(2000)范围内企业的存活比例

### 关键指标

### <font color ='#DE3163'>可以尝试不同窗口大小

In [12]:
def track_business_survival(timepoints_dict):
    """Track business survival across different time points"""
    
    # Baseline: Use 12 months before revaluation
    baseline = timepoints_dict['pre_12m'].dropna(subset=['rateable_value'])
    baseline_uarns = set(baseline['uarn'])
    
    survival_data = []
    
    for period, df in timepoints_dict.items():
        if len(df) == 0:
            continue
            
        df_clean = df.dropna(subset=['rateable_value'])
        current_uarns = set(df_clean['uarn'])
        
        # Calculate survival rate
        survived = len(baseline_uarns & current_uarns)
        survival_rate = survived / len(baseline_uarns) * 100
        
        # Analyze by threshold groups
        for threshold in [15000]:

            near_threshold = baseline[
                (baseline['rateable_value'] >= threshold-2000) & 
                (baseline['rateable_value'] <= threshold+2000)
            ]
            near_threshold_uarns = set(near_threshold['uarn'])
            
            survived_near = len(near_threshold_uarns & current_uarns)
            survival_rate_near = survived_near / len(near_threshold_uarns) * 100 if len(near_threshold_uarns) > 0 else 0
            
            survival_data.append({
                'period': period,
                'threshold': threshold,
                'total_baseline': len(baseline_uarns),
                'near_threshold_baseline': len(near_threshold_uarns),
                'survived_total': survived,
                'survived_near_threshold': survived_near,
                'survival_rate_total': survival_rate,
                'survival_rate_near_threshold': survival_rate_near
            })
    
    return pd.DataFrame(survival_data)